[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/notebooks/bin2d/bin2d_all_features.ipynb)
# Bin2D — All Features
Interactive 2D image binning: gallery mode, mean/sum modes, crop/pad edge handling, calibration tracking, state persistence, and programmatic export.

In [ ]:
# Install in Google Colab
try:
    import google.colab
    !pip install -q -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ quantem-widget
except ImportError:
    pass  # Not in Colab, skip

In [ ]:
try:
    %load_ext autoreload
    %autoreload 2
    %env ANYWIDGET_HMR=1
except Exception:
    pass  # autoreload unavailable (Colab Python 3.12+)

In [ ]:
import numpy as np
import torch
import quantem.widget
from quantem.widget import Bin2D, profile
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
def make_lattice(size=512, spacing=20.0, sigma=2.5, seed=0):
    """Simulate HAADF-STEM image with hexagonal atomic columns."""
    rng = np.random.default_rng(seed)
    y, x = torch.meshgrid(
        torch.arange(size, device=device, dtype=torch.float32),
        torch.arange(size, device=device, dtype=torch.float32), indexing="ij")
    img = torch.zeros((size, size), device=device)
    for dy in range(-1, int(size / spacing) + 2):
        for dx in range(-1, int(size / spacing) + 2):
            cx = dx * spacing + (dy % 2) * spacing / 2
            cy = dy * spacing * 0.866
            r2 = (x - cx)**2 + (y - cy)**2
            img += torch.exp(-r2 / (2 * sigma**2))
    scan_noise = torch.from_numpy(rng.normal(0, 0.03, (size, size)).astype(np.float32)).to(device)
    poisson_scale = 500.0
    img_np = (img.cpu().numpy() * poisson_scale).clip(0)
    img_noisy = rng.poisson(img_np).astype(np.float32) / poisson_scale
    return img_noisy + scan_noise.cpu().numpy()
print(f"Device: {device}")
print(f"quantem.widget {quantem.widget.__version__}")
profile()

## 1. Single Image — Basic Binning

In [ ]:
image = make_lattice(512, spacing=20, seed=42)
Bin2D(image, bin_factor=4, pixel_size=1.2, title="HAADF — 4× Binning")

## 2. Gallery Mode — Multiple Images

In [ ]:
# Defocus series — same structure, different noise realizations
series = np.stack([make_lattice(256, spacing=18, sigma=2.0 + i * 0.5, seed=i) for i in range(5)])
labels = [f"Defocus {i}" for i in range(5)]
Bin2D(series, bin_factor=2, pixel_size=0.8, labels=labels, title="Defocus Series")

## 3. Mean vs Sum Mode

In [ ]:
# Mean preserves intensity scale, sum accumulates (useful for dose-limited data)
w_mean = Bin2D(image, bin_factor=4, bin_mode="mean", title="Mean Binning")
w_sum = Bin2D(image, bin_factor=4, bin_mode="sum", title="Sum Binning")
print(f"Mean mode — original mean: {image.mean():.2f}, binned mean: {w_mean.get_binned_image().mean():.2f}")
print(f"Sum mode  — original mean: {image.mean():.2f}, binned mean: {w_sum.get_binned_image().mean():.2f}")
w_mean

## 4. Crop vs Pad Edge Handling

In [ ]:
# Non-divisible dimensions: 513×513 with factor 4
odd_image = make_lattice(513, spacing=22, seed=99)
w_crop = Bin2D(odd_image, bin_factor=4, edge_mode="crop", title="513×513 — Crop")
w_pad = Bin2D(odd_image, bin_factor=4, edge_mode="pad", title="513×513 — Pad")
print(f"Crop: {odd_image.shape} → {w_crop.binned_height}×{w_crop.binned_width} (trims extra pixels)")
print(f"Pad:  {odd_image.shape} → {w_pad.binned_height}×{w_pad.binned_width} (zero-pads to next multiple)")
w_crop

## 5. Calibration Tracking

In [ ]:
w = Bin2D(image, bin_factor=2, pixel_size=1.2, title="Calibrated")
print(f"Original pixel size: {w.pixel_size:.2f} Å/px")
print(f"Binned pixel size:   {w.binned_pixel_size:.2f} Å/px")
w.bin_factor = 8
print(f"After 8× bin:        {w.binned_pixel_size:.2f} Å/px")
w.summary()

## 6. State Persistence

In [ ]:
w = Bin2D(image, bin_factor=4, cmap="viridis", log_scale=True, title="Saved State")
state = w.state_dict()
print("State:", state)
# Restore into a new widget
w2 = Bin2D(image, state=state)
print(f"Restored: bin_factor={w2.bin_factor}, cmap={w2.cmap}, log_scale={w2.log_scale}")

## 7. Programmatic Data Extraction

In [ ]:
w = Bin2D(series, bin_factor=4, title="Extract")
# All images binned
all_binned = w.get_binned_data()
print(f"All binned: {all_binned.shape}")
# Single image binned
single = w.get_binned_image(idx=2)
print(f"Single binned (idx=2): {single.shape}")

## 8. Display Options

In [ ]:
Bin2D(image, bin_factor=4, cmap="magma", log_scale=True, auto_contrast=True, title="Magma + Log + Auto-contrast")

## 9. set_image — Replace Data Without Recreating

In [ ]:
w = Bin2D(image, bin_factor=4, cmap="viridis", title="Original")
# Swap in a different image — display settings preserved
new_image = make_lattice(512, spacing=30, seed=7)
w.set_image(new_image)
print(f"Replaced image. cmap still={w.cmap}, bin_factor still={w.bin_factor}")
w

## 10. IO Integration — Load Real Data Files

Bin2D accepts IOResult from `IO.file()` / `IO.folder()` directly. Metadata (title, pixel_size, labels) is auto-extracted. Works with TIFF, EMD, NPY, HDF5, and other formats supported by IO.

In [ ]:
import tempfile, os
from quantem.widget import IO
# Save synthetic images as files to demonstrate IO workflow
tmp = tempfile.mkdtemp()
# Single TIFF-like file (saved as .npy for portability)
np.save(os.path.join(tmp, "haadf_2048.npy"), make_lattice(256, spacing=16, seed=10))
# Load with IO.file() — title auto-extracted from filename
result = IO.file(os.path.join(tmp, "haadf_2048.npy"))
print(f"IO loaded: shape={result.data.shape}, title='{result.title}'")
# Pass IOResult directly to Bin2D — no manual array extraction needed
Bin2D(result, bin_factor=4, title="Loaded via IO.file()")

In [ ]:
# IO.file() with bin_factor — bin at load time
result_binned = IO.file(os.path.join(tmp, "haadf_2048.npy"), bin_factor=2)
print(f"IO bin_factor=2: shape={result_binned.data.shape}")
# Compare: IO-level binning vs widget-level binning
w = Bin2D(result_binned, bin_factor=2, title="IO pre-binned → widget 2× more")
print(f"Total reduction: original→IO 2×→widget 2× = 4× total")
w.summary()

In [ ]:
# IO.folder() — load multiple images as gallery
folder = os.path.join(tmp, "series")
os.makedirs(folder, exist_ok=True)
for i in range(4):
    np.save(os.path.join(folder, f"frame_{i:03d}.npy"), make_lattice(256, spacing=18+i*2, seed=i))
result_folder = IO.folder(folder)
print(f"IO.folder loaded: shape={result_folder.data.shape}, labels={result_folder.labels}")
Bin2D(result_folder, bin_factor=4, title="Gallery from IO.folder()")